In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
n_vx = [[0, 0], [0, 1], [0, 2],
        [1, 0], [1, 1], [1, 2],
        [2, 0], [2, 1], [2, 2]]
n_edge = [(0, 1), (1, 2), 
          (3, 4), (4, 5),
          (6, 7), (7, 8),
          (0, 3), (3, 6),
          (1, 4), (4, 7),
          (2, 5), (5, 8)]
triArea = 0.5

In [ ]:
m, fuseMarkers, fuseSegments = wall_generation.triangulate_channel_walls(n_vx, n_edge, triArea, flags="Y")

In [ ]:
fuseMarkers = [0] * 9

In [ ]:
fuseMarkers[4] = 1

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, np.array(fuseMarkers) == 1, epsilon = 1e-5)

In [ ]:
import periodic_unit_helper

In [ ]:
fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

In [ ]:
# isheet.setRelaxedStiffnessEpsilon(1e-6)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
import time, vis
benchmark.reset()
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.pressure = 1
opts.niter = 200
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, [], opts, callback=cb)
benchmark.report()

In [ ]:
cr.success

In [ ]:
ipu.energy()

In [ ]:
ipu

### Compute periodic volume through meshing

In [ ]:
ipu.getVars()

In [ ]:
ipu.setVars([1., 1., 0., 0., 0., -1., 0., 0., 1., 0., 0., -1., 0., 0., 1., 0., 0.,
       -1., 0., 0., 1., 0., 0., -1., 0, 0])

In [ ]:
ipu.periodicVolume()

In [ ]:
mesh = ipu.sheet.mesh()

In [ ]:
sheet = ipu.sheet

In [ ]:
def get_triangles_of_boundary_edges(edge):
    tri1 = [sheet.getDeformedVtxPosition(edge[0], 0), sheet.getDeformedVtxPosition(edge[1], 0), sheet.getDeformedVtxPosition(edge[1], 1)]
    tri2 = [sheet.getDeformedVtxPosition(edge[0], 0), sheet.getDeformedVtxPosition(edge[1], 1), sheet.getDeformedVtxPosition(edge[0], 1)]
    return [tri1, tri2]

In [ ]:
tris = []
for edge in mesh.boundaryElements():
    tris += get_triangles_of_boundary_edges(edge)

In [ ]:
mesh

In [ ]:
len(tris)

In [ ]:
volume = 0
for tri in tris:
    volume += la.det(tri)

In [ ]:
volume / 6

In [ ]:
mesh.boundaryElements()

In [ ]:
interior_volume = 0
for tri in mesh.elements():
    interior_volume += (la.det([sheet.getDeformedVtxPosition(tri[0], 0), sheet.getDeformedVtxPosition(tri[1], 0), sheet.getDeformedVtxPosition(tri[2], 0)]))

In [ ]:
interior_volume

In [ ]:
for tri in mesh.elements():
    interior_volume -= la.det([sheet.getDeformedVtxPosition(tri[0], 1), sheet.getDeformedVtxPosition(tri[1], 1), sheet.getDeformedVtxPosition(tri[2], 1)])

In [ ]:
interior_volume / 6

### Validate gradient

In [ ]:
fd_perturb = np.random.uniform(-1e-1, 1e1, ipu.numVars())

In [ ]:
ipu.setVars(ipu.getVars() + fd_perturb)

In [ ]:
import fd_validation

In [ ]:
class fd_wrapper:
    def __init__(self, ipu):
        self.ipu = ipu

    def setVars(self, v):
        self.ipu.sheet.setVars(v)
        
    def numVars(self):
        return self.ipu.sheet.numVars()

    def getVars(self):
        return self.ipu.sheet.getVars()

    def energy(self):   return self.ipu.energyPeriodicPressurePotential()
    def gradient(self): return self.ipu.gradientPeriodicPressurePotential()    
    def hessian(self): return self.ipu.hessianPeriodicPressurePotential()

In [ ]:
ipu.sheet.pressure

In [ ]:
periodicPressure = fd_wrapper(ipu)

In [ ]:
periodicPressure.energy(), periodicPressure.gradient(), periodicPressure.hessian()

In [ ]:
fd_validation.gradConvergencePlot(periodicPressure)

In [ ]:
fd_validation.hessConvergencePlot(periodicPressure)

In [ ]:
triplet = ipu.hessianPeriodicPressurePotential()

In [ ]:
triplet.m

In [ ]:
nA = np.zeros((triplet.m , triplet.n))

In [ ]:
for entry in triplet.entries():
    if (entry.i > entry.j):
        print(entry.i, entry.j, entry.v)
    nA[entry.i, entry.j] = entry.v

In [ ]:
nA *= 6 / 10

In [ ]:
nA[3:6, 3:6]

In [ ]:
nA[3:6, 3:6]

In [ ]:
ipu.periodicVolume()

In [ ]:
nA[0:3, 6:9]

In [ ]:
nA[0:3, 6:9]

In [ ]:
nA[6:9, 6:9]

In [ ]:
nA[6:9, 6:9]

In [ ]:

v0_indices = [np.arange(0, 3)]
v0_indices = np.array(v0_indices).flatten()

v1_indices = [np.arange(6, 9)]
v1_indices = np.array(v1_indices).flatten()


v0_star_indices = [np.arange(42, 45)]
v0_star_indices = np.array(v0_star_indices).flatten()




In [ ]:
v0_indices, v1_indices, v0_star_indices


In [ ]:

var_types = ['v0_indices', 'v1_indices', 'v0_star_indices']
var_indices = {'v0_indices': v0_indices,
               'v1_indices': v1_indices, 
               'v0_star_indices': v0_star_indices}


In [ ]:
fd_validation.hessian_convergence_block_plot(fd_wrapper(ipu), var_types, var_indices, testHessVec=False)


### Validate total volume objective

In [ ]:
fd_perturb = np.random.uniform(-1e-3, 1e-3, ipu.numVars())

In [ ]:
ipu.setVars(ipu.getVars() + fd_perturb)

In [ ]:
ipu.energy()

In [ ]:
ipu.energy(energyType = Elastic)

In [ ]:
ipu.energy(energyType = Pressure)

In [ ]:
Pressure = inflation.InflatableSheet.EnergyType.Pressure
Elastic  = inflation.InflatableSheet.EnergyType.Elastic
Full     = inflation.InflatableSheet.EnergyType.Full

In [ ]:
fd_validation.gradConvergencePlot(ipu, customArgs = {"energyType": Full})

In [ ]:
fd_validation.hessConvergencePlot(ipu, customArgs = {"energyType": Full})

In [ ]:
fd_validation.gradConvergencePlot(ipu, customArgs = {"energyType": Pressure})

In [ ]:
fd_validation.hessConvergencePlot(ipu, customArgs = {"energyType": Pressure})

In [ ]:
fd_validation.hessConvergencePlot(ipu.sheet, customArgs = {"energyType": Elastic})

In [ ]:
fd_validation.hessConvergencePlot(ipu.sheet, customArgs = {"energyType": Pressure})